# NB06 — Train SAC (Apple Full-Body) — RTX 5090 Optimized

Train a **SAC** agent on `UnitreeG1PlaceAppleInBowlFullBody-v1` with
automatic entropy tuning. Off-policy learning with **10M replay buffer**.
Same training budget as PPO (NB05) for **fair comparison**.

| Parameter | Value |
|-----------|-------|
| Algorithm | SAC (Soft Actor-Critic) |
| Library | Stable-Baselines3 |
| Policy | MlpPolicy [512, 512] **ReLU** (~790K params) |
| Budget | **2,000,000** env steps (same as NB05 PPO) |
| Buffer | **10,000,000** transitions |
| Batch | **1024** |
| Entropy | `ent_coef="auto"` (automatic tuning) |


## Objectives

1. Create env (single env — SAC is off-policy, uses replay buffer).
2. Configure SB3 SAC with same budget and net_arch as NB05 PPO.
3. Train for TOTAL_ENV_STEPS (2M) with checkpointing every 200K.
4. Save model + learning curve.
5. Quick evaluation (20 deterministic episodes).


## Resources

| Resource | Requirement | Notes |
|----------|-------------|-------|
| GPU | **RTX 5090** (32 GB VRAM) | |
| RAM | 40 GB | 10M replay buffer ~4-8 GB |
| Runtime | ~2-4 hours (2M on RTX 5090) | |


## Imports & Setup


In [ ]:
import sys, os, json, random, time
from pathlib import Path

import numpy as np
import torch
import gymnasium as gym
import matplotlib.pyplot as plt
import pandas as pd

from stable_baselines3 import SAC
from stable_baselines3.common.callbacks import BaseCallback, CheckpointCallback

import mani_skill.envs
from mani_skill.utils.wrappers.gymnasium import CPUGymWrapper

PROJECT_ROOT = Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
from src.envs import UnitreeG1PlaceAppleInBowlFullBodyEnv

ARTIFACTS_DIR = PROJECT_ROOT / "artifacts" / "NB06"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

IS_GPU = torch.cuda.is_available()
DEVICE = "cuda" if IS_GPU else "cpu"
print(f"Device: {DEVICE}")
if IS_GPU:
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")


## Configuration


In [ ]:
CFG = {
    "seed":            42,
    "env_id":          "UnitreeG1PlaceAppleInBowlFullBody-v1",
    "control_mode":    "pd_joint_delta_pos",
    "obs_mode":        "state",
    # ── Training budget (SAME as NB05 PPO for fairness) ──
    "total_env_steps": 2_000_000 if IS_GPU else 20_000,
    "n_envs":          1,   # SAC is off-policy → single env + replay buffer
    # ── SAC Hyperparameters (RTX 5090 optimized) ──
    "learning_rate":   3e-4,     # initial LR (linear decay to 1e-5)
    "lr_end":          1e-5,
    "buffer_size":     10_000_000 if IS_GPU else 50_000,
    "batch_size":      1024,
    "tau":             0.005,
    "gamma":           0.99,
    "ent_coef":        "auto",
    "target_entropy":  "auto",
    "learning_starts": 10_000,
    "train_freq":      1,
    "gradient_steps":  1,
    "net_arch":        [512, 512],   # SAME as NB05 PPO
    "activation_fn":   "ReLU",
    # ── Normalization ──
    "vec_normalize":   True,
    "norm_obs":        True,
    "norm_reward":     True,
    "clip_obs":        10.0,
    # ── Checkpointing ──
    "checkpoint_freq": 200_000,
    # ── Eval ──
    "eval_episodes":   20,
}

SEED = CFG["seed"]
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

with open(ARTIFACTS_DIR / "nb06_config.json", "w") as f:
    json.dump(CFG, f, indent=2)

print("SAC Config (RTX 5090 optimized):")
for k, v in CFG.items():
    print(f"  {k}: {v}")


## Step 1 — Linear Learning Rate Schedule


In [ ]:
def linear_schedule(initial_lr: float, final_lr: float = 1e-5):
    def schedule(progress_remaining: float) -> float:
        return final_lr + (initial_lr - final_lr) * progress_remaining
    return schedule

lr_schedule = linear_schedule(CFG["learning_rate"], CFG["lr_end"])
print(f"LR schedule: {CFG['learning_rate']} → {CFG['lr_end']} (linear decay)")


## Step 2 — Create Environment


In [ ]:
env = gym.make(
    CFG["env_id"],
    num_envs=1,
    obs_mode=CFG["obs_mode"],
    control_mode=CFG["control_mode"],
    render_mode="rgb_array",
)
env = CPUGymWrapper(env)
print(f"Env: obs={env.observation_space.shape}, act={env.action_space.shape}")


## Step 3 — Configure SAC (RTX 5090)


In [ ]:
model = SAC(
    "MlpPolicy",
    env,
    learning_rate=lr_schedule,
    buffer_size=CFG["buffer_size"],
    batch_size=CFG["batch_size"],
    tau=CFG["tau"],
    gamma=CFG["gamma"],
    ent_coef=CFG["ent_coef"],
    target_entropy=CFG["target_entropy"],
    learning_starts=CFG["learning_starts"],
    train_freq=CFG["train_freq"],
    gradient_steps=CFG["gradient_steps"],
    policy_kwargs={
        "net_arch": CFG["net_arch"],
        "activation_fn": torch.nn.ReLU,
    },
    verbose=1,
    seed=SEED,
    device=DEVICE,
)

total_params = sum(p.numel() for p in model.policy.parameters())
print(f"SAC model created: {total_params:,} parameters")
print(f"  net_arch: {CFG['net_arch']}, activation: ReLU")
print(f"  buffer: {CFG['buffer_size']:,}, batch: {CFG['batch_size']}")


## Step 4 — Training Callbacks


In [ ]:
class TrainLogCallback(BaseCallback):
    def __init__(self):
        super().__init__()
        self.episode_rewards = []
        self.episode_lengths = []

    def _on_step(self):
        infos = self.locals.get("infos", [])
        for info in (infos if isinstance(infos, list) else [infos]):
            if isinstance(info, dict) and "episode" in info:
                self.episode_rewards.append(float(info["episode"]["r"]))
                self.episode_lengths.append(int(info["episode"]["l"]))
        return True

log_cb = TrainLogCallback()

ckpt_cb = CheckpointCallback(
    save_freq=max(CFG["checkpoint_freq"] // max(CFG["n_envs"], 1), 1),
    save_path=str(ARTIFACTS_DIR),
    name_prefix="checkpoint",
)
print(f"✅ Callbacks ready (log + checkpoint every {CFG['checkpoint_freq']:,} steps)")


## Step 5 — Train SAC (2M steps)


In [ ]:
print(f"\n{'='*60}")
print(f"  Training SAC — {CFG['total_env_steps']:,} steps")
print(f"  buffer={CFG['buffer_size']:,}, batch={CFG['batch_size']}, device={DEVICE}")
print(f"  net_arch={CFG['net_arch']}, activation=ReLU")
print(f"  LR: {CFG['learning_rate']} → {CFG['lr_end']}")
print(f"{'='*60}")

start_time = time.time()
model.learn(
    total_timesteps=CFG["total_env_steps"],
    callback=[log_cb, ckpt_cb],
    progress_bar=True,
)
elapsed = time.time() - start_time
print(f"\nTraining completed in {elapsed:.1f}s ({elapsed/60:.1f} min)")


## Step 6 — Save Model


In [ ]:
model.save(str(ARTIFACTS_DIR / "sac_apple"))
print(f"✅ Model saved: {ARTIFACTS_DIR / 'sac_apple.zip'}")


## Step 7 — Quick Evaluation (20 episodes)


In [ ]:
eval_env = gym.make(
    CFG["env_id"], num_envs=1, obs_mode="state",
    control_mode=CFG["control_mode"], render_mode="rgb_array",
)
eval_env = CPUGymWrapper(eval_env)

eval_results = []
for ep in range(CFG["eval_episodes"]):
    obs, info = eval_env.reset(seed=ep * 100)
    ep_reward, ep_steps = 0.0, 0
    for step in range(1000):
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = eval_env.step(action)
        ep_reward += float(reward)
        ep_steps += 1
        if terminated or truncated:
            break
    eval_results.append({
        "episode": ep, "total_reward": ep_reward,
        "steps": ep_steps, "success": bool(info.get("success", False)),
    })
    print(f"  Ep {ep+1:2d}: reward={ep_reward:.4f}, steps={ep_steps}")

eval_summary = {
    "mean_reward":     float(np.mean([r["total_reward"] for r in eval_results])),
    "std_reward":      float(np.std([r["total_reward"] for r in eval_results])),
    "success_rate":    float(np.mean([r["success"] for r in eval_results])),
    "training_time_s": elapsed,
    "total_steps":     CFG["total_env_steps"],
}

with open(ARTIFACTS_DIR / "eval_results.json", "w") as f:
    json.dump(eval_summary, f, indent=2)

print(f"\nSAC Eval: mean_reward={eval_summary['mean_reward']:.4f}, "
      f"success={eval_summary['success_rate']:.2%}")
eval_env.close()


## Step 8 — Learning Curve


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ep_rewards = log_cb.episode_rewards
if ep_rewards:
    axes[0].plot(ep_rewards, alpha=0.3, color="steelblue", linewidth=0.5)
    window = min(50, len(ep_rewards))
    if len(ep_rewards) >= window:
        rolling = np.convolve(ep_rewards, np.ones(window)/window, mode="valid")
        axes[0].plot(range(window-1, len(ep_rewards)), rolling,
                     color="darkblue", linewidth=2, label=f"Rolling avg ({window})")
    axes[0].set_title("Episode Rewards During Training")
    axes[0].set_xlabel("Episode")
    axes[0].set_ylabel("Total Reward")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

ep_lengths = log_cb.episode_lengths
if ep_lengths:
    axes[1].plot(ep_lengths, alpha=0.3, color="seagreen", linewidth=0.5)
    axes[1].set_title("Episode Lengths")
    axes[1].set_xlabel("Episode")
    axes[1].set_ylabel("Steps")
    axes[1].grid(True, alpha=0.3)

fig.suptitle(f"NB06 — SAC Training ({CFG['total_env_steps']:,} steps, RTX 5090)",
             fontweight="bold")
fig.tight_layout()
fig.savefig(ARTIFACTS_DIR / "learning_curve.png", dpi=150)
plt.show()

training_log = pd.DataFrame({
    "episode": range(len(ep_rewards)),
    "reward": ep_rewards,
})
training_log.to_csv(ARTIFACTS_DIR / "training_log.csv", index=False)


## Cleanup


In [ ]:
env.close()
print("✅ NB06 SAC Training Complete")


## Artifacts

| File | Description |
|------|-------------|
| `sac_apple.zip` | Trained SAC model |
| `checkpoint_*.zip` | Checkpoints every 200K steps |
| `learning_curve.png` | Training reward curve |
| `training_log.csv` | Per-episode stats |
| `eval_results.json` | Quick eval (20 episodes) |
| `nb06_config.json` | Full config |

## RTX 5090 Optimization Notes

- **[512, 512] ReLU** network — same as NB05 for fairness
- **10M replay buffer** — leverages 40 GB RAM
- **Batch 1024** — efficient gradient computation on RTX 5090
- **Linear LR decay** (3e-4 → 1e-5)
- **learning_starts=10,000** — fill buffer with quality data before training
- **Checkpoints** every 200K steps
- **Fairness**: `total_env_steps=2M` and `net_arch=[512,512]` identical to NB05 PPO
